# Mutation Tracks Tutorial

This tutorial covers **LolliplotTrack** and **DandelionTrack** from the
geneview.genometracks module — lollipop-style and dandelion-style visualizations
for mutations, variants, and methylation sites along protein features.

**Topics covered:**
- `LolliplotTrack` — lollipop plot with circle, pie, pin, flag, pie.stack types
- Per-SNP customization (cex, node labels, shapes, label rotation/color)
- Tanghulu stacking (stacked circles for integer scores)
- Caterpillar layout (top/bottom placement)
- Aligned labels with anti-overlap (`jitter="label"`)
- Legend, custom y-axis, coordinate rescaling, multi-layer features
- `DandelionTrack` — clustered variant visualization
- Composing mutation tracks with other genome tracks

> Ported from R/Bioconductor [trackViewer](https://bioconductor.org/packages/trackViewer).

## Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from geneview.genometracks import (
    LolliplotTrack, DandelionTrack,
    lolliplot, dandelion_plot,
    GenomeAxisTrack, AnnotationTrack,
    GenomicInterval, plot_tracks,
)

## Sample Data

We'll use a set of variant positions along a gene with protein domain annotations.

In [ ]:
np.random.seed(42)
SNP = [10, 100, 105, 108, 400, 410, 420, 600, 700, 805, 840, 1400, 1402]
palette = ["#0080FF", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9"]

snp_data = pd.DataFrame({
    "chrom": ["chr1"] * len(SNP),
    "start": SNP,
    "label": [f"snp{s}" for s in SNP],
    "score": np.random.randint(1, 8, len(SNP)),
    "fill": [palette[i % len(palette)] for i in range(len(SNP))],
})

features = pd.DataFrame({
    "chrom": ["chr1", "chr1", "chr1"],
    "start": [1, 501, 1001],
    "end": [120, 900, 1405],
    "name": ["Domain A", "Kinase", "DNA Binding"],
    "fill": ["#FF8833", "#51C6E6", "#DFA32D"],
    "height": [0.04, 0.06, 0.08],
})

region = GenomicInterval("chr1", 0, 1500)
print(f"Variants: {len(snp_data)}")
print(f"Features: {len(features)}")
snp_data.head()

## 1. Basic Lolliplot

In [ ]:
ax = lolliplot(snp_data, features=features, figsize=(12, 4))
plt.title("Basic Lolliplot")
plt.tight_layout()
plt.show()

## 2. Shape Types

Available types: `circle` (default), `pie`, `pin`, `flag`, `pie.stack`.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 16))
for ax, t in zip(axes, ["circle", "pie", "pin", "flag"]):
    df = snp_data.copy()
    if t == "pie":
        df["pie_values"] = [[3, 7]] * len(df)
        df["pie_colors"] = [["#87CEFA", "#98CE31"]] * len(df)
    track = LolliplotTrack(df, features=features, type=t)
    ax.set_xlim(region.start, region.end)
    track.draw(ax, region)
    ax.set_title(f"Type: {t}", fontsize=10, fontweight="bold")
fig.tight_layout()
plt.show()

## 3. Tanghulu Stacking

When scores are small integers (≤ `lollipop_style_switch_limit`), variants
are drawn as stacked circles — one circle per score unit.

In [ ]:
snp_tanghulu = snp_data.copy()
snp_tanghulu["score"] = np.random.randint(1, 6, len(SNP))
ax = lolliplot(snp_tanghulu, features=features, figsize=(12, 4))
plt.title("Tanghulu Stacking (stacked circles)")
plt.tight_layout()
plt.show()

## 4. Per-SNP Customization

Override global settings per row using optional DataFrame columns.

In [ ]:
snp_custom = snp_data.copy()
snp_custom["cex"] = np.linspace(0.4, 2.0, len(SNP))
snp_custom["label_rotation"] = [0, 30, 45, 60, 90, 0, 30, 45, 60, 90, 0, 30, 45]
snp_custom["label_color"] = ["red", "blue", "green", "orange", "purple",
                              "red", "blue", "green", "orange", "purple",
                              "red", "blue", "green"]
ax = lolliplot(snp_custom, features=features, figsize=(12, 4))
plt.title("Per-SNP cex, label rotation & color")
plt.tight_layout()
plt.show()

## 5. Node Labels

Render text **inside** each shape using the `node_label` column.

In [ ]:
snp_nl = snp_data.copy()
snp_nl["score"] = np.random.randint(1, 6, len(SNP))
snp_nl["node_label"] = [str(s) for s in SNP]
snp_nl["node_label_color"] = "white"
snp_nl["node_label_size"] = 5
ax = lolliplot(snp_nl, features=features, figsize=(12, 4))
plt.title("Node Labels Inside Circles")
plt.tight_layout()
plt.show()

## 6. Caterpillar Layout

Place variants above (`side="top"`) and below (`side="bottom"`) the baseline.

In [ ]:
snp_cat = snp_data.copy()
snp_cat["score"] = np.random.randint(1, 5, len(SNP))
snp_cat["side"] = ["top" if i % 2 == 0 else "bottom" for i in range(len(SNP))]
ax = lolliplot(snp_cat, features=features, figsize=(12, 5))
plt.title("Caterpillar Layout (top/bottom)")
plt.tight_layout()
plt.show()

## 7. Aligned Labels (jitter="label")

All labels are placed at a **uniform y-level** at the top, connected by dashed
lines. Close-position labels are automatically staggered to avoid overlap.

In [ ]:
snp_jit = snp_data.copy()
snp_jit["dashline_col"] = [palette[i % len(palette)] for i in range(len(SNP))]
ax = lolliplot(snp_jit, features=features, jitter="label", figsize=(12, 4))
plt.title("Aligned Labels with Anti-overlap")
plt.tight_layout()
plt.show()

## 8. Legend

In [ ]:
snp_leg = snp_data.copy()
snp_leg["fill"] = ["#87CEFA" if i % 2 == 0 else "#98CE31" for i in range(len(SNP))]
legend = {"labels": ["Wild Type", "Mutant"], "fill": ["#87CEFA", "#98CE31"]}
ax = lolliplot(snp_leg, features=features, legend=legend, figsize=(12, 4))
plt.title("Lolliplot with Legend")
plt.tight_layout()
plt.show()

## 9. Custom Y-axis and ylab

In [ ]:
snp_ya = snp_data.copy()
snp_ya["score"] = np.random.randint(1, 11, len(SNP))
ax = lolliplot(snp_ya, features=features, yaxis=[0, 5, 10], ylab="# evidences",
               figsize=(12, 4))
plt.title("Custom Y-axis Ticks + ylab")
plt.tight_layout()
plt.show()

## 10. Different Shapes Per SNP

Available shapes: `circle`, `square`, `diamond`, `triangle_point_up`, `triangle_point_down`.

In [ ]:
snp_sh = snp_data.copy()
snp_sh["score"] = np.random.randint(1, 6, len(SNP))
snp_sh["shape"] = ["circle", "square", "diamond",
                    "triangle_point_up", "triangle_point_down",
                    "circle", "square", "diamond",
                    "triangle_point_up", "triangle_point_down",
                    "circle", "square", "diamond"]
ax = lolliplot(snp_sh, features=features, figsize=(12, 4))
plt.title("Different Shapes Per SNP")
plt.tight_layout()
plt.show()

## 11. Multi-shape Tanghulu

Use list-valued `shape` and `fill` columns for alternating shapes in stacks.

In [ ]:
snp_ms = pd.DataFrame({
    "chrom": ["chr1"] * 4,
    "start": [100, 400, 700, 1100],
    "score": [3, 4, 2, 5],
    "fill": [["#FF0000", "#0000FF", "#00FF00"],
             ["#FF00FF", "#FFFF00"],
             ["#0080FF", "#E69F00"],
             ["#D55E00", "#CC79A7"]],
    "shape": [["circle", "square", "diamond"],
              ["triangle_point_up", "circle"],
              ["square", "diamond"],
              ["triangle_point_down", "triangle_point_up"]],
})
ax = lolliplot(snp_ms, features=features, figsize=(12, 4))
plt.title("Multi-shape Tanghulu Stack")
plt.tight_layout()
plt.show()

## 12. pie.stack Type

In [ ]:
snp_ps = pd.DataFrame({
    "chrom": ["chr1"] * 6,
    "start": [100, 100, 100, 700, 700, 700],
    "score": [1]*6,
    "pie_values": [[3,7],[5,5],[2,8],[6,4],[1,9],[4,6]],
    "pie_colors": [["#0080FF","#E69F00"]]*6,
    "stack_factor": ["A","B","C","A","B","C"],
    "label": ["p1","p2","p3","p4","p5","p6"],
})
ax = lolliplot(snp_ps, type="pie.stack", figsize=(12, 4))
plt.title("pie.stack Type")
plt.tight_layout()
plt.show()

## 13. Coordinate Rescaling

Remap positions using `(from_start, from_end, to_start, to_end)` tuples.

In [ ]:
rescale_map = [(0, 500, 0, 250), (500, 1500, 300, 900)]
ax = lolliplot(snp_data, features=features, rescale=rescale_map, figsize=(12, 4))
plt.title("Rescale Coordinate Mapping")
plt.tight_layout()
plt.show()

## 14. Multi-layer Features

Use `feature_layer_id` to draw features on separate baselines.

In [ ]:
features_ml = pd.DataFrame({
    "chrom": ["chr1"] * 5,
    "start": [1, 501, 1001, 200, 700],
    "end": [120, 900, 1405, 400, 850],
    "name": ["Domain A", "Kinase", "DNA Bind", "Motif X", "Motif Y"],
    "fill": ["#FF8833", "#51C6E6", "#DFA32D", "#CC79A7", "#009E73"],
    "feature_layer_id": [1, 1, 1, 2, 2],
})
ax = lolliplot(snp_data, features=features_ml, figsize=(12, 5))
plt.title("Multi-layer Features")
plt.tight_layout()
plt.show()

## 15. DandelionTrack

Nearby variants are clustered; each cluster fans out from a central stem.

In [ ]:
np.random.seed(42)
dense = pd.DataFrame({
    "chrom": ["chr1"] * 30,
    "start": np.sort(np.random.randint(10, 1400, 30)),
    "score": np.random.randint(1, 8, 30),
    "label": [f"var{i}" for i in range(30)],
})
ax = dandelion_plot(dense, features=features, figsize=(12, 4))
plt.title("DandelionTrack")
plt.tight_layout()
plt.show()

## 16. Composing with Other Tracks

Combine mutation tracks with GenomeAxisTrack and AnnotationTrack.

In [ ]:
tracks = [
    GenomeAxisTrack(),
    AnnotationTrack(features, name="Domains", shape="box", show_label=True),
    LolliplotTrack(snp_data, features=features, name="Mutations"),
]
axes = plot_tracks(tracks, region=region, figsize=(12, 7),
                   title="Lolliplot + Annotation + Axis")
plt.tight_layout()
plt.show()

## 17. Multi-sample Comparison

Stack multiple LolliplotTracks for side-by-side comparison.

In [ ]:
np.random.seed(123)
sample_b = snp_data.copy()
sample_b["score"] = np.random.randint(1, 8, len(SNP))
sample_b["fill"] = "#DB7575"

tracks = [
    GenomeAxisTrack(),
    LolliplotTrack(snp_data, features=features, name="Sample A"),
    LolliplotTrack(sample_b, features=features, name="Sample B"),
]
axes = plot_tracks(tracks, region=region, figsize=(12, 8),
                   title="Two Samples")
plt.tight_layout()
plt.show()

---

## Further Reading

- [Mutation Tracks Guide](../mutation_tracks_guide.md) — full API reference and parameter tables
- [Genome Tracks Guide](../genome_tracks_guide.md) — general genome track documentation
- [trackViewer vignette](https://jianhong.github.io/trackViewer/articles/lollipopPlot.html) — original R/Bioconductor reference
- Example scripts: `examples/scripts/mutation_lollipop_trackviewer.py`